# 02 · Өгөгдөл цэвэрлэх ба EDA

Түүхий өгөгдөлд **4 асуудал** байна. Тэдгээрийг засаад, дараа нь өгөгдлийн **зүй тогтлыг** судална.

| # | Асуудал | Шийдэл |
|---|---|---|
| 1 | Нийтлэлийн нэр солигдсон | Хуучин нэрийн уншилтыг нэмэх |
| 2 | 0 уншилттай өдөр алга | 0-ээр нөхөх |
| 3 | Нийтлэл "амьд" болохоос өмнөх үе | Хасах |
| 4 | Нэг өдрийн гэнэтийн өсөлт (бот) | Илрүүлж хязгаарлах |

Бүх цэвэрлэгээний код `src/clean.py` дотор байгаа. Энд алхам бүрийг харуулж тайлбарлана.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_white"
BLUE, RED, GRAY = "#2a78d6", "#e63946", "#9aa0ad"   # сайттай ижил өнгө

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import clean

P = ROOT / "data" / "processed"
pv = pd.read_parquet(P / "pageviews.parquet")        # түүхий
alias = pd.read_parquet(P / "alias_pageviews.parquet")
total = pd.read_parquet(P / "total_views.parquet")
print(f"Түүхий: {len(pv):,} мөр, {pv['article'].nunique()} сэдэв")

Түүхий: 586,420 мөр, 301 сэдэв


## 1. Нэр солигдсон нийтлэл

Wikipedia-д нийтлэлийн нэр солигдоход хуучин нэрийн уншилт **шинэ нэрэнд тооцогддоггүй**.
Жишээ нь "J. D. Vance" 2024 онд "JD Vance" болсон. Засахгүй бол model "2024 он хүртэл хэн ч сонирхоогүй" гэж буруу сурна.

In [2]:
merged = clean.merge_aliases(pv, alias)

# JD Vance: засахаас өмнө ба дараа (сарын дундаж)
before = pv[pv["article"] == "JD_Vance"].set_index("date")["views"].resample("MS").mean()
after = merged[merged["article"] == "JD_Vance"].set_index("date")["views"].resample("MS").mean()
df = pd.DataFrame({"Засахаас өмнө": before, "Засварын дараа": after}).reset_index()
fig = px.line(df, x="date", y=["Засахаас өмнө", "Засварын дараа"], log_y=True,
              color_discrete_sequence=[RED, BLUE], title="JD Vance: өдрийн дундаж уншилт (log)")
fig.update_layout(legend_title=None, yaxis_title="уншилт / өдөр", xaxis_title=None)
fig.show()

## 2. Алга болсон өдрүүд

Wikimedia API **0 уншилттай өдрийг огт буцаадаггүй**. Тиймээс тэр өдрүүд "дутуу" мэт харагдана.
Энэ нь мэдээлэл алдагдсан биш, үнэхээр хэн ч уншаагүй гэсэн үг тул **0-ээр нөхнө**
(interpolation хийвэл байхгүй уншилтыг зохиож оруулах болно).

In [3]:
end = pv["date"].max()
filled = clean.fill_days(merged, end)
print(f"Нөхсөн өдөр: {int(filled['was_missing'].sum()):,}  (бүх мөрийн {filled['was_missing'].mean():.2%})")
filled[filled["was_missing"]].groupby("category").size().rename("нөхсөн өдөр").to_frame()

Нөхсөн өдөр: 4,545  (бүх мөрийн 0.77%)


,нөхсөн өдөр
category,
film_tv,71
games_anime,5
tech_ai,14
viral,4455


## 3. Нийтлэл "амьд" болохоос өмнөх үе

Зарим нийтлэл үүсэхээсээ өмнө эсвэл **redirect** байх үедээ 0 уншилттай. Ийм үе model-ийг төөрөгдүүлнэ.

Дүрэм: 7 хоногийн дундаж уншилт **t өдөр хүртэлх 1 жилийн медианы 2%** эсвэл **өдөрт 100**-д хүрсэн өдрөөс эхлэн авна.
Медианыг зөвхөн **өнгөрсөн** өдрүүдээр тооцно (ADR 0007). Ирээдүйн медиан ашиглавал "вирал болохоос өмнөх нам гүм үе"
хасагдаж, бидний таамаглах гэж буй үе алга болно (look-ahead). Тиймээс бараг зөвхөн бүхэлдээ 0 уншилттай эхлэл хасагдана.

In [4]:
trimmed, report = clean.trim_inactive(filled)
print(f"Хассан өдөр: {report['removed_days'].sum():,}")
report.sort_values("removed_days", ascending=False).head(8)

Хассан өдөр: 1,247


,article,start,removed_days
173,Lokah,2024-04-21,1202
2,2022_FIFA_World_Cup_knockout_stage,2021-02-10,37
93,Extreme_Rules_(2021),2021-01-09,8
204,Natarajan_Chandrasekaran,2021-01-01,0
203,Narendra_Modi,2021-01-01,0
202,Nag_Ashwin,2021-01-01,0
201,Naadam,2021-01-01,0
200,NATO,2021-01-01,0


In [5]:
g = filled[filled["article"] == "Lokah"]
start = report.set_index("article").loc["Lokah", "start"]
fig = px.line(g, x="date", y="views", log_y=True, color_discrete_sequence=[BLUE],
              title="Lokah: шар тасархай шугамаас өмнөх 0 уншилттай үеийг хассан")
fig.add_vline(x=start, line_dash="dash", line_color="#f0a500")
fig.update_layout(yaxis_title="уншилт / өдөр (log)", xaxis_title=None)
fig.show()

## 4. Нэг өдрийн гэнэтийн өсөлт (outlier detection)

**Жинхэнэ шуугиан** хэдэн өдөр үргэлжилдэг: оргилын дараа аажмаар буурдаг.
**Бот** ихэвчлэн ганц өдөр гэнэт гарч ирээд маргааш нь алга болдог.

Дүрэм: тухайн өдөр 29 хоногийн медианаас **10+ дахин** их, гэвч өмнөх ба дараагийн өдөр нь **2 дахинаас бага** байвал
"тусгаарлагдсан өсөлт" гэж үзээд хөршүүдийнх нь утгаар сольно. Олон өдөр үргэлжилсэн шуугианд хүрэхгүй.

In [6]:
capped = clean.cap_isolated_spikes(trimmed)
sp = capped[capped["is_spike"]]
print(f"Илэрсэн тусгаарлагдсан өсөлт: {len(sp)}  ({sp['article'].nunique()} сэдэвт)")
sp.sort_values("views_raw", ascending=False)[["date", "article", "views_raw", "views"]].head(8)

Илэрсэн тусгаарлагдсан өсөлт: 39  (33 сэдэвт)


,date,article,views_raw,views
469624,2026-08-10,Roblox,4781328,11863
369308,2024-11-12,Microsoft,502957,14348
469671,2026-09-26,Roblox,370335,9985
147415,2022-02-17,Donald_Trump,370155,27072
469590,2026-07-07,Roblox,160323,7773
64763,2024-06-15,BTS,140144,8579
343624,2021-06-20,Louis_XIV,89218,7303
548086,2022-10-11,Travis_Kelce,84503,8639


In [7]:
r = capped[(capped["article"] == "Roblox") & (capped["date"].between("2026-06-01", "2026-10-31"))]
fig = px.line(r, x="date", y=["views_raw", "views"], log_y=True, color_discrete_sequence=[RED, BLUE],
              title="Roblox: улаан = түүхий (ганц өдрийн 4.8 сая бот), цэнхэр = цэвэрлэсэн")
fig.for_each_trace(lambda t: t.update(name={"views_raw": "түүхий", "views": "цэвэрлэсэн"}[t.name]))
fig.update_layout(legend_title=None, yaxis_title="уншилт / өдөр (log)", xaxis_title=None)
fig.show()

## 5. Target: анхаарлын хувь

Wikipedia-ийн нийт уншилт жил ирэх тусам буурч байгаа (01 notebook). Тиймээс тоог биш **хувийг** таамаглана:

$$\text{attention} = \frac{\text{сэдвийн уншилт}}{\text{тухайн өдрийн нийт уншилт}} \times 10^6 \qquad y = \log(1 + \text{attention})$$

**Яагаад log?** Уншилт маш тэгш бус тархсан: ихэнх өдөр бага, хааяа маш их. Log нь энэ "урт сүүл"-ийг шахаж,
model-д том ба жижиг сэдвийг ижил хэмжээсээр харьцуулах боломж олгоно.

In [8]:
d = clean.add_attention(capped, total)
smp = d.sample(8_000, random_state=0)
cut = smp["attention"].quantile(0.99)   # графикт уншигдахуйц болгох үүднээс зөвхөн 99%-ийг харуулна
fig1 = px.histogram(smp[smp["attention"] <= cut], x="attention", nbins=60, color_discrete_sequence=[RED],
                    title="Өмнө: attention — ихэнх нь 0-д ойр, урт сүүлтэй (тэгш бус)")
fig2 = px.histogram(smp, x="y", nbins=60, color_discrete_sequence=[BLUE],
                    title="Дараа: y = log1p(attention) — урт сүүл алга болж, тэгш тархсан")
fig1.show(); fig2.show()

In [9]:
# Цэвэрлэсэн өгөгдлийг хадгална (src/clean.py яг энэ алхмуудыг хийнэ)
clean_df = clean.build()
print(f"clean.parquet: {len(clean_df):,} мөр, {clean_df['article'].nunique()} сэдэв")

clean.parquet: 589,772 мөр, 301 сэдэв


# EDA: өгөгдлийн зүй тогтол

Эндээс олсон зүй тогтол бүр маргааш (03) **feature** болно.

## A. Ангилал бүрийн ердийн анхаарал

In [10]:
cat_mn = {"politics": "Улс төр", "film_tv": "Кино", "true_crime": "True crime", "tech_ai": "AI", "sports": "Спорт",
          "music": "Хөгжим", "games_anime": "Тоглоом", "celebs": "Алдартнууд", "society": "Нийгэм",
          "mongolia": "Монгол", "viral": "Вирал"}
c = clean_df.assign(cat=clean_df["category"].astype(str).map(cat_mn))
order = c.groupby("cat")["y"].median().sort_values().index.tolist()
fig = px.box(c.sample(8_000, random_state=1), x="y", y="cat", category_orders={"cat": order},
             color_discrete_sequence=[BLUE], points=False, title="Ангилал бүрийн log(анхаарал) тархалт")
fig.update_layout(yaxis_title=None, xaxis_title="y = log1p(attention)")
fig.show()

> **Олдвор:** Алдартнууд, спорт, хөгжим хамгийн их анхаарал татдаг; Монгол ба вирал сэдвүүд ердийн өдөр бага.
> Вирал сэдэв ердийн үедээ бага, гэнэт өсдөг → **ангилал** чухал feature.

## B. Гарагийн хэв маяг

In [11]:
w = c[c["date"] > c["date"].max() - pd.Timedelta(days=730)].copy()
w["dow"] = w["date"].dt.dayofweek
idx = w.groupby(["cat", "dow"])["y"].mean().unstack()
idx = idx.sub(idx.mean(axis=1), axis=0)            # ангилал бүрийн дунджаас зөрүү
idx.columns = ["Да", "Мя", "Лх", "Пү", "Ба", "Бя", "Ня"]
fig = px.imshow(idx, color_continuous_scale=[BLUE, "#ffffff", RED], color_continuous_midpoint=0, aspect="auto",
                title="Гарагийн хэв маяг: улаан = дунджаас их, цэнхэр = бага (сүүлийн 2 жил)")
fig.update_layout(coloraxis_colorbar_title="зөрүү")
fig.show()

> **Олдвор:** AI, улс төр, нийгмийг ажлын өдөр; спорт, кино, true crime-ийг амралтын өдөр илүү уншдаг.
> → **гараг (dow)** ба **амралтын өдөр эсэх** feature.

## C. Улирлын давталт

In [12]:
h = clean_df[clean_df["article"].isin(["Halloween", "Christmas"])].copy()
h["year"] = h["date"].dt.year.astype(str)
h["doy"] = h["date"].dt.dayofyear
fig = px.line(h[h["year"].isin(["2022", "2023", "2024", "2025"])], x="doy", y="y", color="year", facet_col="article",
              title="Halloween ба Christmas: жил бүр яг ижил өдөр оргилдоо хүрдэг")
fig.update_layout(xaxis_title="жилийн хэддэх өдөр", yaxis_title="y")
fig.show()

> **Олдвор:** Баярын сэдвүүд жил бүр давтагддаг → **жилийн өдөр (sin/cos)** ба **сар** feature.

## D. Autocorrelation: өнгөрсөн нь ирээдүйг хэр таамагладаг вэ?

Autocorrelation гэдэг нь өнөөдрийн утга N өдрийн өмнөх утгатайгаа хэр төстэй вэ гэдгийг 0–1-ээр хэмждэг.

In [13]:
lags = list(range(1, 29))
acf = []
for a, g in clean_df.groupby("article", observed=True):
    s = g["y"].reset_index(drop=True)
    if len(s) > 200:
        acf.append([s.autocorr(l) for l in lags])
acf = np.nanmean(np.array(acf), axis=0)
fig = px.bar(x=lags, y=acf, color=[("7 хоног" if l % 7 == 0 else "бусад") for l in lags],
             color_discrete_map={"7 хоног": RED, "бусад": BLUE},
             title="Сэдвүүдийн дундаж autocorrelation: өчигдөр хамгийн хүчтэй, аажмаар буурна")
fig.update_layout(xaxis_title="хэдэн өдрийн өмнө (lag)", yaxis_title="autocorrelation", legend_title=None)
fig.show()

> **Олдвор:** Өчигдрийн утгатай хамаарал маш өндөр (~0.91), 14 хоногийн өмнөхтэй ч ~0.64 хэвээр.
> Өөрөөр хэлбэл сонирхол "удаан санадаг". Долоо хоногийн мөчлөг дундаж дээр бараг харагдахгүй ч
> зарим ангилалд (B график) байгаа. → **lag_1, lag_2, lag_7, lag_14** ба **rolling дундаж** feature.

## E. Вирал сэдэв хэр хурдан мартагддаг вэ?

In [14]:
curves = []
for a, g in clean_df[clean_df["category"] == "viral"].groupby("article", observed=True):
    s = g.set_index("date")["views"]
    seg = s[s.idxmax(): s.idxmax() + pd.Timedelta(days=14)]
    if len(seg) == 15:
        curves.append(seg.values / s.max() * 100)
decay = np.median(np.array(curves), axis=0)
fig = px.bar(x=list(range(15)), y=decay, color_discrete_sequence=[RED],
             title=f"Вирал сэдвийн оргилын дараах анхаарал (медиан, {len(curves)} сэдэв): 3 хоногт {decay[3]:.0f}% үлддэг")
fig.update_layout(xaxis_title="оргилоос хойш хоног", yaxis_title="оргилын %")
fig.show()

> **Олдвор:** Шуугиан маш хурдан унтардаг. → **оргилоос хойш хэдэн хоног** ба **өсөлтийн хурд** feature.
> Мөн ирэх 3 өдрийн таамаг яагаад хэрэгтэйг харуулна: 3 хоногт бүх зүйл өөрчлөгддөг.

## ✅ Дүгнэлт

| Цэвэрлэгээ | Үр дүн |
|---|---|
| Нэр солигдсон | 3 нийтлэл засагдсан |
| Дутуу өдөр | 0-ээр нөхсөн |
| "Амьд" болохоос өмнөх | 0 уншилттай эхлэлийг хассан (causal, ADR 0007) |
| Бот | Ганц өдрийн тусгаарлагдсан өсөлтийг хязгаарласан |

**Маргааш (03) хийх feature-үүд:** ангилал, гараг, амралтын өдөр, сар, жилийн өдөр (sin/cos),
lag 1/7/14, rolling дундаж/хэлбэлзэл, өсөлтийн хурд, оргилоос хойш хэдэн хоног.